# Лабораторная работа 4: Градиентный бустинг (HAR UCI)

**Задание:**
- Использовать датасет Human Activity Recognition Using Smartphones (UCI).
- Обучить модель градиентного бустинга с обоснованием гиперпараметров.
- Посчитать `accuracy`, `precision`, `recall`, `F1`.
- Построить ROC-кривую.

In [ ]:
# Если библиотек нет, раскомментируйте установку:
# %pip install -U scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
)
from sklearn.preprocessing import label_binarize

sns.set(style="whitegrid")
np.random.seed(42)

## 1) Загрузка данных HAR UCI

Ожидаем, что архив HAR UCI распакован локально и содержит папку `UCI HAR Dataset`.

Используем стандартное разделение `train/test`, предоставленное авторами датасета.

In [ ]:
# Укажите путь к папке с датасетом при необходимости
base_dir = Path("UCI HAR Dataset")

X_train_path = base_dir / "train" / "X_train.txt"
y_train_path = base_dir / "train" / "y_train.txt"
X_test_path = base_dir / "test" / "X_test.txt"
y_test_path = base_dir / "test" / "y_test.txt"

features_path = base_dir / "features.txt"
activity_labels_path = base_dir / "activity_labels.txt"

# Загрузка
X_train = pd.read_csv(X_train_path, delim_whitespace=True, header=None)
X_test = pd.read_csv(X_test_path, delim_whitespace=True, header=None)
y_train = pd.read_csv(y_train_path, header=None).squeeze("columns")
y_test = pd.read_csv(y_test_path, header=None).squeeze("columns")

features = pd.read_csv(features_path, delim_whitespace=True, header=None, names=["idx", "feature"])
activity_labels = pd.read_csv(activity_labels_path, delim_whitespace=True, header=None, names=["id", "activity"])

X_train.columns = features["feature"].values
X_test.columns = features["feature"].values

label_map = dict(zip(activity_labels["id"], activity_labels["activity"]))

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Classes:", sorted(y_train.unique()))
activity_labels

## 2) Обучение Gradient Boosting и обоснование гиперпараметров

Выбран `GradientBoostingClassifier` со следующими настройками:
- `n_estimators=250`: достаточное количество слабых деревьев для сложной зависимости в 561 признаке.
- `learning_rate=0.05`: более аккуратное обновление, меньше риск переобучения.
- `max_depth=3`: неглубокие деревья обычно дают хороший баланс смещения/дисперсии.
- `subsample=0.8`: стохастический бустинг для повышения устойчивости.

Поскольку задача многоклассовая (6 активностей), считаем метрики в `weighted`-варианте.

In [ ]:
gb_clf = GradientBoostingClassifier(
    n_estimators=250,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    random_state=42,
)

gb_clf.fit(X_train, y_train)

y_pred = gb_clf.predict(X_test)
y_proba = gb_clf.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted")
rec = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")

target_names = [label_map[i] for i in sorted(label_map.keys())]
print("\nClassification report:\n")
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
# Матрица ошибок
cm = confusion_matrix(y_test, y_pred, labels=sorted(label_map.keys()))
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names,
)
plt.title("Confusion Matrix (Gradient Boosting, HAR UCI)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ROC-кривые One-vs-Rest для многоклассовой классификации
classes = sorted(label_map.keys())
y_test_bin = label_binarize(y_test, classes=classes)

fpr = {}
tpr = {}
roc_auc = {}

for i, cls in enumerate(classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Микро-усреднённая ROC
fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), y_proba.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

plt.figure(figsize=(8, 6))
for i, cls in enumerate(classes):
    plt.plot(
        fpr[i],
        tpr[i],
        lw=1.7,
        label=f"{label_map[cls]} (AUC = {roc_auc[i]:.3f})",
    )

plt.plot(
    fpr["micro"],
    tpr["micro"],
    color="black",
    linestyle="--",
    lw=2,
    label=f"micro-average (AUC = {roc_auc['micro']:.3f})",
)
plt.plot([0, 1], [0, 1], color="gray", linestyle=":" , label="Random classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for HAR Classification (Gradient Boosting)")
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

## 3) Вывод

В работе:
- загружен и проанализирован датасет HAR UCI;
- обучен `GradientBoostingClassifier` с обоснованными гиперпараметрами;
- рассчитаны `accuracy`, `precision`, `recall`, `F1`;
- построены ROC-кривые по схеме one-vs-rest.

Для дальнейшего улучшения можно выполнить подбор гиперпараметров (`RandomizedSearchCV`) и сравнить с `XGBoost`/`LightGBM`.